# TrainLM on TPU

This is the same Hugging Face-like workflow an end user runs. Choose a TPU runtime, fill in the four values below, and call `trainer.train()`.

TrainLM handles TPU discovery, world size, worker launch, ranks, preflight, caching, distributed data ownership, checkpoints, evaluation, and structured results under the hood.

## 1. Install

Run from a checked-out TrainLM repository, then restart the notebook kernel.

In [ ]:
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt

## 2. Your inputs

Choose how many numbered shards to use. `(0, TRAIN_SHARD_STOP)` is end-exclusive, so `2` downloads shards `00000` and `00001`. The next shard is reserved for evaluation. TrainLM resolves `main` to its immutable Hub commit before downloading.


In [ ]:
MODEL_ID = "microsoft/Phi-3.5-mini-instruct"
MODEL_REVISION = "PUT_THE_40_CHARACTER_HF_COMMIT_SHA_HERE"
DATASET_ID = "LaughTaleAI/LaughLM-Tokenized-Fine"
DATASET_REVISION = "main"
TRAIN_SHARD_STOP = 2
OUTPUT_DIR = Path("/kaggle/working/trainlm-run")


## 3. Build the datasets and trainer

`from_hub()` downloads the requested `.bin` range through the Hugging Face cache, scans every token, validates the legacy header and vocabulary bounds, and creates the private manifests needed by TPU workers. Users do not download files or configure a dataloader, process count, rank, or world size.


In [ ]:
from trainlm import PackedBinDataset, TrainLMTrainer, TrainLMTrainingArguments

sequence_length = 2048
train_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=DATASET_REVISION,
    shard_range=(0, TRAIN_SHARD_STOP),
    sequence_length=sequence_length,
    split="train",
)
eval_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=train_dataset.hub_revision,
    shard_range=(TRAIN_SHARD_STOP, TRAIN_SHARD_STOP + 1),
    sequence_length=sequence_length,
    split="validation",
)

trainer = TrainLMTrainer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=TrainLMTrainingArguments(
        output_dir=OUTPUT_DIR,
        accelerator="tpu",
        bf16=True,
        max_steps=6,
        sequence_length=sequence_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        logging_steps=1,
        eval_steps=2,
        save_steps=2,
    ),
)


## 4. Train

This single call performs the private collective probe and model preflight, launches every available TPU worker, trains, evaluates every two steps, and writes committed checkpoints every two steps.

In [ ]:
result = trainer.train()
result

## 5. Optional: inspect what TrainLM selected

`explain()` is useful when reviewing fallbacks or filing a result. It does not require users to inspect worker commands or logs.

In [ ]:
trainer.explain(format="text")

## 6. Optional: resume

Normally set this to the last committed checkpoint after an interrupted or completed run. TrainLM validates topology and restores model, optimizer, scheduler, runtime, RNG, trainer, and packed-data position internally.

In [ ]:
resumed_trainer = TrainLMTrainer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=TrainLMTrainingArguments(
        output_dir="/kaggle/working/trainlm-resumed",
        accelerator="tpu",
        bf16=True,
        max_steps=10,
        sequence_length=sequence_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        logging_steps=1,
        eval_steps=2,
        save_steps=2,
    ),
)
resumed_result = resumed_trainer.train(
    resume_from_checkpoint=OUTPUT_DIR / "checkpoint-4"
)
resumed_result

## What to save from a validation run

Archive the output directory, including `coordinator_summary.json`, `summary.json`, `metrics.jsonl`, committed checkpoint manifests/shards, and XLA metrics. The returned result is the normal user-facing status; these files are only needed for debugging or performance certification.

A successful run validates the lifecycle on that TPU. It does not by itself mark performance as certified.